# NB1 — Population Pre-training

**Goal:** Train EEGNet on 101 subjects (LOSO), save one checkpoint per fold.  
These checkpoints are the *population prior* that NB2 fine-tunes on per-subject calibration data.

**Architecture:** EEGNet (braindecode 1.2.0) — learns temporal + spatial filters end-to-end.  
Unlike Riemannian MDM, it sees full (64-channel × 321-timepoint) epochs — time is preserved.

**Preprocessing:**
- Bandpass 1–79 Hz (same as NB8 broadband)
- Euclidean Alignment (EA) per subject: `R⁻¹² @ X` centres each subject's covariance at identity
- EA here uses each subject's *own* full data (all 3 runs).  
  NB2 will re-derive EA from calibration runs only (clean evaluation).

**Expected runtime:** ~2–3 h on GTX 1080 Ti (102 folds × ~90 s/fold).

In [1]:
import os, gc, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.linalg import eigh

import mne
mne.set_log_level('WARNING')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from braindecode.models import EEGNet

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results',     exist_ok=True)

PyTorch 2.5.1+cu121  |  device: cuda
  GPU: NVIDIA GeForce GTX 1080 Ti


In [2]:
# ── Data ──────────────────────────────────────────────────────────────────────
TASK2_RUNS  = [4, 8, 12]        # imagined left / right fist
SFREQ       = 160               # Hz
TMIN, TMAX  = 1.0, 3.0         # post-cue epoch window
N_TIMES     = int((TMAX - TMIN) * SFREQ) + 1   # 321 (MNE includes both endpoints)
N_CHANS     = 64
BANDHI      = 79.0              # Hz

KNOWN_BAD   = {88, 89, 92, 100, 104, 106}  # from original neuro-data-explorer series

# ── EEGNet ────────────────────────────────────────────────────────────────────
# braindecode defaults match the original EEGNet paper for this data size
F1            = 8
D             = 2
F2            = 16
KERNEL_LENGTH = 64      # temporal filter  (~0.4 s at 160 Hz)
DROP_PROB     = 0.25

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS      = 150
BATCH_SIZE  = 64
LR          = 1e-3
WDECAY      = 1e-4

print(f'N_TIMES={N_TIMES}  N_CHANS={N_CHANS}  KERNEL_LENGTH={KERNEL_LENGTH}')
print(f'Training: {EPOCHS} epochs  lr={LR}  batch={BATCH_SIZE}')

N_TIMES=321  N_CHANS=64  KERNEL_LENGTH=64
Training: 150 epochs  lr=0.001  batch=64


In [3]:
def load_subject(sub):
    """Load, bandpass, epoch one subject. Returns (X µV, y) or (None, None)."""
    paths = mne.datasets.eegbci.load_data(sub, runs=TASK2_RUNS, verbose=False)
    raws  = [mne.io.read_raw_edf(p, preload=True, verbose=False) for p in paths]
    raw   = mne.concatenate_raws(raws)
    mne.datasets.eegbci.standardize(raw)
    raw.filter(1., BANDHI, verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)
    motor_ids = {k: v for k, v in event_id.items() if k in ('T1', 'T2')}
    if not motor_ids:
        return None, None

    epochs = mne.Epochs(raw, events, event_id=motor_ids,
                        tmin=TMIN, tmax=TMAX, baseline=None,
                        preload=True, verbose=False)
    X = epochs.get_data(units='uV').astype(np.float32)   # (n_ep, 64, 321)
    y = (epochs.events[:, 2] == motor_ids.get('T2', -1)).astype(np.int64)
    return X, y


print('Loading all subjects …  (first run may download ~1 GB from PhysioNet)')
t0         = time.time()
raw_store  = {}   # sub -> (X µV, y)
failed     = []

for sub in range(1, 110):
    try:
        X, y = load_subject(sub)
        if X is None or len(y) < 10:
            failed.append(sub)
        else:
            raw_store[sub] = (X, y)
    except Exception:
        failed.append(sub)

print(f'Loaded {len(raw_store)} subjects in {time.time()-t0:.0f}s')
if failed:
    print(f'Failed to load: {failed}')

Loading all subjects …  (first run may download ~1 GB from PhysioNet)


Loaded 106 subjects in 78s
Failed to load: [88, 92, 100]


In [4]:
# Artifact screen: flag subjects whose median peak-to-peak amplitude
# is > 2.5 SD above the group mean (same criterion as original series)
ptp_vals   = {s: float(np.median(np.ptp(raw_store[s][0], axis=-1)))
              for s in raw_store}
ptp_arr    = np.array(list(ptp_vals.values()))
ptp_thresh = ptp_arr.mean() + 2.5 * ptp_arr.std()

auto_bad   = {s for s, v in ptp_vals.items() if v > ptp_thresh}
flagged    = auto_bad | KNOWN_BAD
SUBS       = sorted(s for s in raw_store if s not in flagged)

print(f'PTP threshold : {ptp_thresh:.1f} µV')
print(f'Auto-flagged  : {sorted(auto_bad)}')
print(f'Known bad     : {sorted(KNOWN_BAD & set(raw_store))}')
print(f'Clean N       : {len(SUBS)}')

PTP threshold : 406.3 µV
Auto-flagged  : [32]
Known bad     : [89, 104, 106]
Clean N       : 102


In [5]:
def ea_invsqrt(X):
    """
    Euclidean Alignment: compute R^{-1/2} from (n_ep, n_ch, n_t) signal.
    R = mean epoch-covariance; eigh gives the symmetric square-root inverse
    that centres each subject's covariance at identity before pooling.
    """
    n_ep, n_ch, n_t = X.shape
    R = np.mean(
        [X[i].astype(np.float64) @ X[i].astype(np.float64).T / n_t
         for i in range(n_ep)],
        axis=0
    )
    w, v = eigh(R)
    w    = np.maximum(w, 1e-10)
    return ((v * (w ** -0.5)) @ v.T).astype(np.float32)


def apply_ea(X, R_invsqrt):
    """Whiten signal: X_aligned[i] = R^{-1/2} @ X[i]  (spatial op, preserves time)"""
    return np.einsum('ij,njt->nit', R_invsqrt.astype(np.float64),
                     X.astype(np.float64)).astype(np.float32)


# Pre-compute EA matrices AND pre-aligned data for all subjects.
# Storing pre-aligned arrays (~37 MB) avoids 10K redundant einsum calls in the LOSO loop.
# Note: uses *all* runs. NB2 re-derives EA from calibration runs only (no leakage).
print('Computing EA matrices + pre-aligned data …')
ea_mats  = {}   # sub -> R^{-1/2}  (64x64)
ea_data  = {}   # sub -> (X_aligned, y)

for sub in SUBS:
    X, y          = raw_store[sub]
    R             = ea_invsqrt(X)
    ea_mats[sub]  = R
    ea_data[sub]  = (apply_ea(X, R), y)

print(f'Done — {len(ea_mats)} subjects  |  one aligned array: {ea_data[SUBS[0]][0].shape}')

Computing EA matrices + pre-aligned data …


Done — 102 subjects  |  one aligned array: (45, 64, 321)


In [6]:
def make_eegnet():
    return EEGNet(
        n_chans           = N_CHANS,
        n_outputs         = 2,
        n_times           = N_TIMES,
        final_conv_length = 'auto',
        F1                = F1,
        D                 = D,
        F2                = F2,
        kernel_length     = KERNEL_LENGTH,
        drop_prob         = DROP_PROB,
    ).to(DEVICE)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, n = 0., 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        n          += len(yb)
    return total_loss / n


@torch.no_grad()
def eval_acc(model, X, y):
    model.eval()
    xt    = torch.FloatTensor(X).to(DEVICE)
    preds = model(xt).argmax(1).cpu().numpy()
    return float((preds == y).mean() * 100.)


# Smoke test — verify shape before the 2-hour LOSO run
_m  = make_eegnet()
_x  = torch.randn(4, N_CHANS, N_TIMES).to(DEVICE)
assert _m(_x).shape == (4, 2), 'shape mismatch'
n_params = sum(p.numel() for p in _m.parameters())
del _m, _x; gc.collect(); torch.cuda.empty_cache()
print(f'EEGNet smoke-test passed  |  params: {n_params:,}')

EEGNet smoke-test passed  |  params: 2,450


In [7]:
loso_acc = {}   # sub -> float accuracy

hdr = f'{"Sub":>4} | {"N_tr":>5} | {"N_te":>5} | {"Acc%":>5} | {"Run mean":>8} | {"Fold t":>6}'
print(hdr)
print('-' * len(hdr))

t_total = time.time()

for test_sub in SUBS:
    ckpt = f'checkpoints/fold_{test_sub:03d}.pt'

    # Resume support: if checkpoint exists, load the stored accuracy and skip training.
    if os.path.exists(ckpt) and os.path.exists('results/loso_results.npy'):
        saved = dict(np.load('results/loso_results.npy', allow_pickle=True).item())
        if test_sub in saved:
            loso_acc[test_sub] = saved[test_sub]
            print(f'{test_sub:>4} | (skipped — checkpoint exists  acc={saved[test_sub]:.1f}%)')
            continue

    train_subs = [s for s in SUBS if s != test_sub]

    # ── Pooled training set (pre-aligned per subject) ─────────────────────────
    X_tr = np.concatenate([ea_data[s][0] for s in train_subs])
    y_tr = np.concatenate([ea_data[s][1] for s in train_subs])

    # ── Test set ──────────────────────────────────────────────────────────────
    X_te, y_te = ea_data[test_sub]

    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
        batch_size = BATCH_SIZE,
        shuffle    = True,
        drop_last  = True,
        pin_memory = (DEVICE == 'cuda'),
    )

    # ── Train ─────────────────────────────────────────────────────────────────
    model     = make_eegnet()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WDECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    t_fold = time.time()
    for _ in range(EPOCHS):
        train_epoch(model, loader, optimizer, criterion)
        scheduler.step()

    # ── Evaluate + checkpoint ─────────────────────────────────────────────────
    acc = eval_acc(model, X_te, y_te)
    torch.save(model.state_dict(), ckpt)
    loso_acc[test_sub] = acc

    del model; gc.collect(); torch.cuda.empty_cache()

    # Persist partial results so the resume logic above works on interruption
    np.save('results/loso_results.npy', loso_acc)

    run_mean = np.mean(list(loso_acc.values()))
    elapsed  = time.time() - t_fold
    print(f'{test_sub:>4} | {len(X_tr):>5} | {len(X_te):>5} | {acc:>5.1f} | {run_mean:>8.2f} | {elapsed:>5.0f}s')

print('-' * len(hdr))
accs = np.array(list(loso_acc.values()))
print(f'\nLOSO mean ± std : {accs.mean():.2f} ± {accs.std():.2f}%')
print(f'Median          : {np.median(accs):.2f}%')
print(f'Min / Max       : {accs.min():.1f} / {accs.max():.1f}%')
print(f'Total time      : {(time.time()-t_total)/60:.1f} min')

np.save('results/loso_results.npy', loso_acc)
print('Final results saved → results/loso_results.npy')

 Sub |  N_tr |  N_te |  Acc% | Run mean | Fold t
------------------------------------------------


   1 |  4545 |    45 |  68.9 |    68.89 |   284s


   2 |  4545 |    45 |  86.7 |    77.78 |   292s


   3 |  4545 |    45 |  75.6 |    77.04 |   290s


   4 |  4545 |    45 |  86.7 |    79.44 |   290s


...

 103 |  4545 |    45 |  80.0 |    72.99 |   265s


 105 |  4545 |    45 |  91.1 |    73.18 |   263s


 107 |  4545 |    45 |  62.2 |    73.07 |   265s


 108 |  4545 |    45 |  88.9 |    73.22 |   263s


 109 |  4545 |    45 |  64.4 |    73.14 |   278s
------------------------------------------------

LOSO mean ± std : 73.14 ± 12.40%
Median          : 73.33%
Min / Max       : 35.6 / 97.8%
Total time      : 1278.2 min
Final results saved → results/loso_results.npy


In [8]:
# Load saved results (run this cell standalone after LOSO completes)
loso_acc  = dict(np.load('results/loso_results.npy', allow_pickle=True).item())
SUBS_used = sorted(loso_acc.keys())
accs      = np.array([loso_acc[s] for s in SUBS_used])

INK   = '#1A1A1A'
OX    = '#002147'
SEPIA = '#704214'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.patch.set_facecolor('white')

# -- Distribution histogram ---------------------------------------------------
ax1.hist(accs, bins=20, color=OX, edgecolor='white', alpha=0.85)
ax1.axvline(accs.mean(),  color=SEPIA, lw=2,   label=f'Mean {accs.mean():.1f}%')
ax1.axvline(50,           color=INK,   lw=1.2, ls='--', alpha=0.4, label='Chance')
ax1.set_xlabel('LOSO accuracy (%)', color=INK)
ax1.set_ylabel('Subjects',          color=INK)
ax1.set_title('NB1 Population model — accuracy distribution', color=INK, fontsize=11)
ax1.legend(frameon=False)
ax1.set_facecolor('white')

# -- Sorted bars (skill profile) ---------------------------------------------
sorted_accs = np.sort(accs)
bar_colors  = [OX if a >= 62 else (SEPIA if a >= 50 else '#C0C0C0')
               for a in sorted_accs]
ax2.bar(range(len(sorted_accs)), sorted_accs, color=bar_colors, width=0.9)
ax2.axhline(accs.mean(), color=SEPIA, lw=1.5, ls='--', alpha=0.8)
ax2.axhline(50,          color=INK,   lw=1,   ls=':',  alpha=0.5)
ax2.set_xlabel('Subject rank (low → high)',   color=INK)
ax2.set_ylabel('LOSO accuracy (%)',            color=INK)
ax2.set_title(f'Sorted accuracy  N={len(accs)}  (mean {accs.mean():.1f}%  median {np.median(accs):.1f}%)',
              color=INK, fontsize=11)
ax2.set_ylim(30, 100)
ax2.set_facecolor('white')

plt.tight_layout()
plt.savefig('results/fig_nb1_loso.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/fig_nb1_loso.png')

Saved → results/fig_nb1_loso.png


## Summary

| | NB9 (neuro-data-explorer) | NB1 (this) |
|---|---|---|
| Channels | 19 (motor ROI) | 64 (full cap) |
| N subjects | 75 (learning curve peak) | 102 (full LOSO) |
| EA | per-subject | per-subject |
| Architecture | EEGNet | EEGNet |
| LOSO accuracy | 65.1% | **see above** |

**What changes going into NB2:**
- Each fold's checkpoint becomes the *population prior*
- Fine-tune on calibration runs 1–2 of the held-out subject (30 epochs ≈ 30 labelled trials per class)
- Evaluate on run 3 (never seen during pre-training *or* fine-tuning)
- EA for fine-tuning re-derived from calibration runs only — no leakage

**Checkpoints saved:** `checkpoints/fold_NNN.pt` — one per test subject.  
**Results saved:** `results/loso_results.npy`